In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_28.pickle

In [3]:
%%RecordEvent
%%time
### cell 28 ###

# Optimized code for CPU and memory efficiency

def grab_subset_of_data_40(original_df, question_of_interest):
    # Select only columns containing the question and map names to choice text
    subset = original_df.filter(like=question_of_interest, axis=1)
    mapping = {col: col.rsplit('-', 1)[-1].strip() for col in subset.columns}
    return subset.rename(columns=mapping).dropna(how='all', subset=mapping.values())


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
    question_of_interest,
    include_2017=False
):
    # Cleaning patterns and lambdas
    question_old = 'Which of the following hosted notebook products do you use on a regular basis?'
    question_new = 'Do you use any of the following hosted notebook products?'
    colab_old_2019 = 'Google Colab '
    kaggle_old_2019 = 'Kaggle Notebooks (Kernels) '
    colab_old_2018 = 'Google Colab'
    kaggle_old_2018 = 'Kaggle Kernels'
    colab_new = 'Colab Notebooks'
    kaggle_new = 'Kaggle Notebooks'
    clean_Q = lambda col: col.replace(question_old, question_new)
    clean_2019 = lambda col: (
        col.replace(colab_old_2019, colab_new)
           .replace(kaggle_old_2019, kaggle_new)
           .replace(question_old, question_new)
    )
    clean_2018 = lambda col: (
        col.replace(colab_old_2018, colab_new)
           .replace(kaggle_old_2018, kaggle_new)
           .replace(question_old, question_new)
    )
    # Prepare list of (raw_df, cleaning_fn, year)
    sources = [
        (responses_df_2018_cell10, clean_2018, '2018'),
        (responses_df_2019_cell10, clean_2019, '2019'),
        (responses_df_2020,        clean_Q,     '2020'),
        (responses_df_2021,        clean_Q,     '2021'),
        (responses_df_2022_cell10, lambda x: x, '2022'),
    ]
    if include_2017:
        sources.insert(0, (responses_df_2017, lambda x: x, '2017'))

    dflist = []
    years = []
    for df_src, col_fn, yr in sources:
        df_clean = df_src.rename(columns=col_fn)
        df_sub = grab_subset_of_data_40(df_clean, question_of_interest)
        df_sub['year'] = yr
        dflist.append(df_sub)
        years.append(yr)

    df_combined = pd.concat(dflist, ignore_index=True)
    # Count non-null per choice for each year without groupby
    counts = [df_sub.drop(columns='year').count() for df_sub in dflist]
    df_combined_counts = pd.DataFrame(counts, index=years).reset_index().rename(columns={'index': 'year'})
    return df_combined, df_combined_counts


def convert_df_of_counts_to_percentages_40(df, df_counts):
    dfp = df_counts.set_index('year')
    sizes = df['year'].value_counts()
    dfp = dfp.div(sizes, axis=0).mul(100)
    return dfp.reset_index()

# Execution
question_of_interest_cell40 = 'Do you use any of the following hosted notebook products?'
notebooks_df_combined, notebooks_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest_cell40
    )
notebooks_df_combined_percentages = convert_df_of_counts_to_percentages_40(
    notebooks_df_combined,
    notebooks_df_combined_counts
)
notebooks_df_combined_percentages_cell40 = notebooks_df_combined_percentages.loc[
    :, ['year', 'None', 'Kaggle Notebooks', 'Colab Notebooks']
]
df_cell40 = notebooks_df_combined_percentages_cell40.melt(
    id_vars=['year'],
    value_vars=['None', 'Kaggle Notebooks', 'Colab Notebooks']
)
df_cell40 = df_cell40.rename(columns={'variable': ''})
df_cell40.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    15 non-null     object 
 1           15 non-null     object 
 2   value   12 non-null     float64
dtypes: float64(1), object(2)
memory usage: 488.0+ bytes
CPU times: user 442 ms, sys: 63.7 ms, total: 506 ms
Wall time: 506 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_28_try_4.pickle